# nb_08_silver_claim_payments

**Layer**: Silver | **Source**: `lh_bronze_insurance.claim_payments_raw` | **Target**: `lh_silver_insurance.claim_payments`

**Data Quality Rules**:
- Cast `payment_amount` to DoubleType; `payment_date` to DateType
- Require non-null: `payment_id`, `claim_id`, `payment_amount`
- Deduplicate on `payment_id` (keep latest `_ingested_at`)
- Standardize `payment_method` to lowercase
- Rejects → `claim_payments_quarantine`

**Idempotency**: Uses `overwrite` mode.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, to_date, current_timestamp, lit, row_number, when, coalesce, lower
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType

try:
    spark
except NameError:
    spark = SparkSession.builder.appName("nb_08_silver_claim_payments").getOrCreate()

print("nb_08_silver_claim_payments - Silver Layer Transformation")

In [ ]:
# ============================================================
# Step 1: Read from Bronze
# ============================================================
df_bronze = spark.table("claim_payments_raw")
bronze_count = df_bronze.count()
print(f"Bronze rows read: {bronze_count:,}")
df_bronze.printSchema()

In [ ]:
# ============================================================
# Step 2: Cast types & clean
# ============================================================
df_typed = df_bronze \
    .withColumn("payment_id", trim(col("payment_id"))) \
    .withColumn("claim_id", trim(col("claim_id"))) \
    .withColumn("payment_date", to_date(col("payment_date"), "yyyy-MM-dd")) \
    .withColumn("payment_amount", col("payment_amount").cast(DoubleType())) \
    .withColumn("payment_method", lower(trim(col("payment_method"))))

print("Type casting complete.")

In [ ]:
# ============================================================
# Step 3: Validate & route rejects
# ============================================================
REQUIRED_FIELDS = ["payment_id", "claim_id"]

rejection_conditions = [
    when((col(f).isNull()) | (col(f) == ""), lit(f"null_{f}"))
    for f in REQUIRED_FIELDS
]
# Numeric field: null after cast means original was empty
rejection_conditions.append(
    when(col("payment_amount").isNull(), lit("null_payment_amount"))
)

df_validated = df_typed.withColumn("_rejection_reason", coalesce(*rejection_conditions))

df_valid = df_validated.filter(col("_rejection_reason").isNull()).drop("_rejection_reason")
df_rejected = df_validated.filter(col("_rejection_reason").isNotNull())

valid_count = df_valid.count()
rejected_count = df_rejected.count()
print(f"Valid: {valid_count:,} | Rejected: {rejected_count:,}")

In [ ]:
# ============================================================
# Step 4: Deduplicate on payment_id
# ============================================================
window_spec = Window.partitionBy("payment_id").orderBy(col("_ingested_at").desc())

df_deduped = df_valid \
    .withColumn("_row_num", row_number().over(window_spec)) \
    .filter(col("_row_num") == 1) \
    .drop("_row_num")

deduped_count = df_deduped.count()
dupes_removed = valid_count - deduped_count
print(f"After dedup: {deduped_count:,} ({dupes_removed} duplicates removed)")

In [ ]:
# ============================================================
# Step 5: Write Silver & Quarantine
# ============================================================
silver_columns = ["payment_id", "claim_id", "payment_date",
                  "payment_amount", "payment_method"]

df_silver = df_deduped.select(silver_columns) \
    .withColumn("_processed_at", current_timestamp())

df_silver.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("claim_payments")
print(f"✓ {deduped_count:,} rows → claim_payments")

if rejected_count > 0:
    df_rejected.withColumn("_quarantined_at", current_timestamp()) \
        .write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable("claim_payments_quarantine")
    print(f"✓ {rejected_count:,} rows → claim_payments_quarantine")

In [ ]:
# ============================================================
# Validation
# ============================================================
print("\nSILVER CLAIM PAYMENTS - SUMMARY")
print("=" * 50)
print(f"  Bronze input:       {bronze_count:>8,}")
print(f"  Rejected:           {rejected_count:>8,}")
print(f"  Duplicates removed: {dupes_removed:>8,}")
print(f"  Silver output:      {deduped_count:>8,}")
print(f"  Quality rate:       {(deduped_count/bronze_count*100):.1f}%")

assert spark.table("claim_payments").count() == deduped_count
print("\n✓ Verification passed")
spark.table("claim_payments").show(5, truncate=25)